# Bivariate analysis


The objective of this notebook is to analyze each variable vs. the target to understand patterns, associations, and potential causal relationships between the variables.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import mannwhitneyu, ks_2samp
import statsmodels.api as sm
from statsmodels.formula.api import ols
import category_encoders as ce

import os
import warnings
import pickle
from IPython.display import Image

from src.utils import load_from_pickle, save_to_pickle

warnings.filterwarnings("ignore")

images_directory = 'images/images_bivar_analysis'
os.makedirs(images_directory, exist_ok=True)

pickle_directory = 'variables/variables_bivar_analysis'
os.makedirs(pickle_directory, exist_ok=True)

In [ ]:
data = pd.read_parquet('../data/silver/df_fraud_univar.parquet', engine= 'fastparquet')
data.head()

## step & day_of_month vs. isFraud

### Step

#### Statistical tests

Is there a difference in the measures or distributions of the variable 'step' between the two groups defined by the target variable? I can plot it, sure, but I will also apply some statistical analysis to represent the difference between both groups.


I use the Mann-Whitney U Test because the variable 'step' doesn't follow a normal distribution. This test is suitable for comparing the central tendency of step between fraudulent and non-fraudulent transactions.

In [ ]:
group1 = data[data['isFraud'] == 'fraud']['step']
group2 = data[data['isFraud'] == 'no_fraud']['step']

stat, p_value = mannwhitneyu(group1, group2)

print(f"Mann-Whitney U Statistic: {stat}")
print(f"P-Value: {p_value}")

- Mann-Whitney U Statistic: 34829829709'5. The U statistic measures the difference between distributions. The bigger the U, the greater the difference but this value doesn't show anything by itself. I need to consider it after paying attention to the p-value.

- p-value = 0 so there is a significative statistical difference of 'step' between the two catories of 'isFraud'. The variable 'step' behaves differently in each group.

---

But as we saw in the univariate analysis our independent variable is VERY unbalanced and this can affect statistical results with any test:

isFraud
- no_fraud    6354407
- fraud          8213

isFraud
- no_fraud    99.871%
- fraud        0.129%

Multiple tests increase the robustness of our conclusions. If different tests lead to similar results, I can be more confident in my findings. The Kolmogorov-Smirnov Test can be useful when we can't assume normality and it's suitable for comparing the overall distribution of step between fraudulent and non-fraudulent transactions.

In [ ]:
ks_stat, ks_p_value = ks_2samp(group1, group2)

print(f"KS Statistic: {ks_stat}")
print(f"P-Value: {ks_p_value}")

- KS Statistic = 0.38 means that there is a difference of the 38% in some point of the CDF (Cumulative Distribution Function) = clear discrepancy between distributions

- p-value = 0, same conclusion as before

#### Summary stats

In [ ]:
data.groupby('isFraud')['step'].describe()


#### Visualization

In [ ]:
image_filename = 'lineplot_step_vs_fraud.png'
plot_filepath = os.path.join(images_directory, image_filename)

if not os.path.isfile(plot_filepath):
    total_value_counts = data['step'].value_counts().sort_index()
    fraud_value_counts = data[data['isFraud'] == 'fraud']['step'].value_counts().sort_index()
    non_fraud_value_counts = data[data['isFraud'] == 'no_fraud']['step'].value_counts().sort_index()

    fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(15, 20), sharex=True)

    titles = [
        "Frequency of Steps in Total Transactions", 
        "Frequency of Steps in Fraudulent Transactions", 
        "Frequency of Steps in Non-Fraudulent Transactions"
    ]
    colors = ['#00008B', '#8B0000', '#006400']
    counts_list = [total_value_counts, fraud_value_counts, non_fraud_value_counts]

    for ax, counts, title, color in zip(axes, counts_list, titles, colors):
        ax.plot(counts.index, counts, marker='o', color=color, linestyle='-', markersize=6)
        ax.set_title(title)
        ax.set_xlabel("Step")
        ax.set_ylabel("Frequency")
        ax.grid(True, which='both', linestyle='--', linewidth=0.7)
        ax.set_xticks(np.arange(min(counts.index), max(counts.index) + 1, 24)) 
        ax.set_xticks(np.arange(min(counts.index), max(counts.index) + 1, 1), minor=True)  
        ax.tick_params(axis='x', which='minor', length=3, color='grey')

    plt.tight_layout()
    plt.savefig(plot_filepath)  
    plt.close(fig)  

Image(filename=plot_filepath)

In [ ]:
#TODO - fix x axis and x labels (de 24 en 24)

In [ ]:
image_filename = 'boxplot_step_vs_fraud.png'
plot_filepath = os.path.join(images_directory, image_filename)

if not os.path.isfile(plot_filepath):
    plt.figure(figsize=(10, 6))  
    sns.boxplot(x='isFraud', y='step', data=data, palette='pastel')
    plt.title('Box Plot of Step by isFraud')

    plt.tight_layout()
    plt.savefig(plot_filepath)  
    plt.close()  

Image(filename=plot_filepath)

Observations:

For context, 1 step represents 1 hour. The total range of hour data is equivalent to a month. 

Non-fraudulent transactions vary over time, fraudulent transactions remain constant.


Non-fraudulent transactions happen inside defined step ranges and slow down when the step is a bit higher than 400 (15 days aprox). 
- The first step range makes sense since people get their salaries at the beginning of the month. 
- The second step range, as I said in the univariate analysis notebook doesn't make sense to me. In this case it would be helpful to know, for example, which month of the year it is to understand if there is any special event in the middle of the month that made people use more transactions.
- There is a gap between the first 48 and 120 hours. It would be interesting to get more information about the month, year, country from where the transaction was sent/received. Was it a bank holiday so people would be on holidays? 

Fraudulent transactions stay constant along the step range. That means fraudsters behaviour stays constant all over the month. 
- There is also a step with very high frequency between the steps 193 and 217.
- Fraudsters usually take very short time to process the transaction so it would be also nice to obtain data about the amount of time spent on the merchants website, the time used by the client to confirm the transaction (short times can indicate that a bot is actually using the platform), etc.

---



#### Additional analysis

What is happening in the very frequent step? How is the relationship between that step and fraud?

In [ ]:
contingency_table = pd.crosstab(data['step'], data['isFraud'])
contingency_table['total'] = contingency_table['fraud'] + contingency_table['no_fraud']
contingency_table['fraud_proportion'] = (contingency_table['fraud'] / contingency_table['total']) * 100

contingency_table.loc[(contingency_table.index > 193) & (contingency_table.index < 220)]

Observations:

There are far more fraudulent transactions on the step 212. 212 steps = 8 days + 20h.
- Why so many people are commiting fraud that day? Why at that time? Was it a coincidence? 
- Was it Black Friday or maybe another special day where a website makes offers?
Time to ask what could have happened here from a business perspective and check the characteristics of those transactions.


The are also other picks on the fraudulent transactions that could be analysed along the gaps of days between them even though the amount of fraud transactions are very constant over time. 


----

Are night transactions more likely to be fraudulent? 

In [ ]:
time_data = data[['step', 'day_of_month', 'isFraud']].copy()
time_data['hour_of_day'] = data['step'] % 24
time_data

In [ ]:
count_hours = time_data.groupby(['hour_of_day']).agg(
    count_fraud=pd.NamedAgg(column='isFraud', aggfunc=lambda x: (x == 'fraud').sum()),
    count_no_fraud=pd.NamedAgg(column='isFraud', aggfunc=lambda x: (x == 'no_fraud').sum())
    
).reset_index()

count_hours    

In [ ]:
plt.figure(figsize=(8, 5))
sns.lineplot(data=count_hours, x='hour_of_day', y='count_fraud', marker='o')
plt.xlabel('Step')
plt.ylabel('Count of Fraud')
plt.title('Average of Fraud over Time Steps')
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.lineplot(data=count_hours, x='hour_of_day', y='count_no_fraud', marker='o')
plt.xlabel('Step')
plt.ylabel('Count of Fraud')
plt.title('Average of Fraud over Time Steps')
plt.grid()
plt.show()

Observations: 

The frequency of fraudulent transactions stay constant, but in the non-fraudulent transactions goes down during the night.

---


### Day of the Month

#### Statistical tests

I don't perform statistical tests because the transformation applied to the variable 'step' to become 'day_of_the_moth' slightly changes the distribution but not enough to be considered normal (check skewness, kurtosis and the plot in the following section)

#### Summary stats


In [ ]:
data.groupby('isFraud')['day_of_month'].describe()

#### Visualization

In [ ]:
contingency_table = time_data.groupby(['day_of_month', 'isFraud'], observed=True).size().unstack(fill_value=0)
proportions = contingency_table.div(contingency_table.sum(axis=1), axis=0) * 100

plt.figure(figsize=(12, 6))
proportions.plot(kind='bar', stacked=True, color=['#ff9999', '#66b3ff'], ax=plt.gca())
plt.title('Proportions of Fraud and No Fraud by Day of Month', fontsize=16)
plt.xlabel('Day of Month', fontsize=14)
plt.ylabel('Proportion (%)', fontsize=14)
plt.xticks(rotation=45)
plt.legend(title='isFraud', title_fontsize='13', fontsize='11')
plt.tight_layout()

plt.show()

Observations:

- The last day of the month has only fraudulent transactions. Curious, right? I would need more info about the context but it looks like it's a coincidence.
- In the same days where non fraudulent transactions (remember we had the hypothesis that it would may be )
Besides that I don't see any clear pattern.

#### Additional analysis

In [ ]:
contingency_table = pd.crosstab(data['day_of_month'], data['isFraud'])
contingency_table['total'] = contingency_table['fraud'] + contingency_table['no_fraud']
contingency_table['fraud_proportion'] = (contingency_table['fraud'] / contingency_table['total']) * 100
contingency_table

Observations:

- I can see the same pattern again, the days 3,4,5 vary from the rest. I assume this could be a bank holiday or another special day but I don't have enough information, like the month of the year, to explain that behavior. 

- Again the last day of the month only has fraudulent transactions.

## type vs. isFraud

#### Statistical tests

In [ ]:
contingency_table = pd.crosstab(data['type'], data['isFraud'])
contingency_table

ANOVA test converting the variable 'isFraud' to numerical since chi-square test needs higher frequency in the categories

In [ ]:
df = data.copy()
df['isFraud'] = df['isFraud'].replace({'no_fraud':0, 'fraud':1})
df['isFraud'] = df['isFraud'].astype(int) 
df

In [ ]:
model = ols('isFraud ~ C(type)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table

Observations:

- The large F-statistic (5539.856588) supports the conclusion that the variability between the means of different type categories is significantly greater than the variability within the groups.

- The p-value is extremely small (effectively 0), indicating that the differences in the means of isFraud across different type categories are statistically significant.

#### Summary stats

In [ ]:
data.groupby('isFraud')['type'].describe()

Observations:

The only types that have fraudulent transactions are CASH_OUT and TRANSFER and as we saw in the univariate analysis they are substantially less common

#### Visualization

In [ ]:
contingency_table = data.groupby(['type', 'isFraud'], observed=True).size().unstack(fill_value=0)
proportions = contingency_table.div(contingency_table.sum(axis=1), axis=0) * 100

plt.figure(figsize=(12, 6))
proportions.plot(kind='bar', stacked=True, color=['#ff9999', '#66b3ff'], ax=plt.gca())
plt.title('Proportions of Fraud and No Fraud by Type', fontsize=16)
plt.xlabel('Type', fontsize=14)
plt.ylabel('Proportion (%)', fontsize=14)
plt.xticks(rotation=45)
plt.legend(title='isFraud', title_fontsize='13', fontsize='11')
plt.tight_layout()

plt.show()

In [ ]:
contingency_table

Observations:

- CASH_IN, DEBIT, PAYMENT have 0 fraud records. 
- CASH_OUT means withdrawing money from the ATM, probably by copying your credit card, a typical fraud strategy I even suffered.
- TRANSFER means that the fraudsters had access to the personal data from the customers? I need more business information to understand the way this fraud is done.

## amount_range & amount vs. isFraud

### amount_range

#### Statistical tests

In [ ]:
model = ols('isFraud ~ C(amount_range)', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table

Observations:

- The large F-statistic (8922.379697) supports the conclusion that the variability between the means of different type categories is significantly greater than the variability within the groups.

- The p-value is extremely small (effectively 0), indicating that the differences in the means of isFraud across different type categories are statistically significant.


#### Summary stats

In [ ]:
data.groupby('isFraud')['amount_range'].describe()

Observations:

- The range of 100.000-1.000.000 has most of both fraudulent and non-fraudulent transactions 

In [ ]:
contingency_table = pd.crosstab(data['amount_range'], data['isFraud'])
contingency_table

#### Visualization

In [ ]:
plt.figure(figsize=(10, 6))
segmented_bar = sns.histplot(
    data=data,
    x="amount_range", 
    hue="isFraud", 
    multiple="fill", 
    shrink=0.8,
    palette = 'pastel'
    
)
plt.xticks(rotation=45)
segmented_bar.set(ylabel="Proportion")
plt.title('Segmented Bar Chart of Fraud by Amount Range')
plt.show()

Observations:

From the crosstab, we can see that the amount ranges between 10k and 10M are the ones with most fraudulent transactions but the proportions show that highest values will have more probability to be fraudulent.

We have to remember that bins have not the same size

### amount

##### Statistical tests

In [ ]:
group1 = data[data['isFraud'] == 'fraud']['amount']
group2 = data[data['isFraud'] == 'no_fraud']['amount']

stat, p_value = mannwhitneyu(group1, group2)

print(f"Mann-Whitney U Statistic: {stat}")
print(f"P-Value: {p_value}")

- p-value = 0 so there is a significative statistical difference of 'amount' between the two catories of 'isFraud'. The variable 'amount' behaves differently in each group.

- Mann-Whitney U Statistic: 41224999623.5. The range between both groups is very huge. 


##### Summary stats

In [ ]:
data.groupby('isFraud')['amount'].describe()

Observations:

- The average amount of fraudulent transactions is much higher than that of non-fraudulent transactions so fraudulent transactions tend to involve larger amounts of money.

- The standard deviation is also much higher for fraudulent transactions, the presence of some very high-value fraudulent transactions.

- The 25th percentile value for fraudulent transactions is significantly higher, indicating that even the smaller fraudulent transactions tend to involve more money than the non-fraudulent transactions.

- The median amount for fraudulent transactions is substantially higher than for non-fraudulent transactions. Half of the fraudulent transactions are below approximately 441,423, while half of the non-fraudulent transactions are below approximately 74,685.

- The 75th percentile value for fraudulent transactions is much higher, indicating that a significant portion of fraudulent transactions are of very high value compared to non-fraudulent ones.

- Non-fraudulent transactions are far more numerous and have a wider range of amounts, but their typical transaction amounts are much smaller than those of fraudulent transactions.

##### Visualization

In [ ]:
contingency_table_amounts = pd.crosstab(data['amount'], data['isFraud'])
contingency_table_amounts

In [ ]:
contingency_table_amounts[contingency_table_amounts['fraud'] > 2]

Observations: 

- I have the hypothesis that humans usually work with rounded values. There is a huge number of fraud transactions moving 1.000000e+07€. It would be interesting to continue doing research on transactions with rounded amounts.

In [ ]:
image_filename = 'boxplot_amount_vs_fraud.png'
plot_filepath = os.path.join(images_directory, image_filename)
    
if not os.path.isfile(plot_filepath):
    fig = plt.figure(figsize=(12, 6))
    boxplot = sns.boxplot(x='isFraud', y='amount', data=data, palette='pastel')
    plt.title('Boxplot of Amount by Fraud Status')
    plt.xlabel('Fraud Status')
    plt.ylabel('Amount')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(plot_filepath)  
    plt.close(fig)  


Image(filename=plot_filepath)

##### Additional analysis

Which are the characteristics of the transactions with amount = 0?

In [ ]:
data.query('amount == 0')

Observations:

Withrawals with amount = 0 are fraudulent, none of them is repeated and they have been used to try to withdraw money. We could analyze the origin and destination users and see their behaviour. One example could be using the postal code, the IP data, the account age, etc. to find hidden patterns.

---

Is there a more frequent amount of money that fraudsters prefer to work with?

In [ ]:
pickle_filename = 'repeated_amounts.pkl'
pickle_filepath = os.path.join(pickle_directory, pickle_filename)

is_cached = False
try:
    repeated_amounts = load_from_pickle(pickle_filepath)
    is_cached = True
except FileNotFoundError:
    is_cached = False

if is_cached:
    print("The DataFrame is already stored in cache")
else:
    print("DataFrame not found. Calculating...")

    repeated_rows = data[data.duplicated('amount', keep=False) & (data['amount'] != 0)]

    repeated_amounts = (repeated_rows.groupby('amount')
                        .agg(count_is_not_Fraud=('isFraud', lambda x: (x == 'no_fraud').sum()),
                             count_isFraud=('isFraud', lambda x: (x == 'fraud').sum()),
                             percentage_is_not_Fraud=('isFraud', lambda x: round((x == 'no_fraud').mean() * 100, 2)),
                             percentage_isFraud=('isFraud', lambda x: round((x == 'fraud').mean() * 100, 2)))
                        .reset_index())

    save_to_pickle(repeated_amounts, pickle_filepath)

repeated_amounts

In [ ]:
repeated_amounts.query("count_isFraud > 2")


Most amounts are repeated 2 times. There are only 3 that are repeated more than 2 times. Further analysis could be focused on analysing the characteristics of those accounts. 

---
I have another question that I will leave for a possible further analysis:

- Since humans are more likely to understand and work with integer amounts, are they more prone to reflect fraudulent transactions? 


## nameOrig vs. isFraud

### Statistical tests

In [ ]:
print(f'The number of unique values in the column is: {data["nameOrig"].nunique()}')
print(f'The total length of the column is: {len(data["nameOrig"])}')


Observations:

There are too many unique values in this column. It does not make sense to test the column.

In [ ]:
data['nameOrig'].value_counts()

### Summary stats

In [ ]:
data.groupby('isFraud')['nameOrig'].describe()

Observations:

- Frequency in fraudulent transactions shows that accounts are always used once.

In [ ]:
contingency_table = pd.crosstab(data['nameOrig'], data['isFraud'])
contingency_table

### Additional analysis

In [ ]:
nameOrig_counts = data['nameOrig'].value_counts()

nameOrig_gt3 = nameOrig_counts[nameOrig_counts == 3].index
subset_3 = data[data['nameOrig'].isin(nameOrig_gt3)]
subset_3.head()

Observations:
    
- nameOrig = 3 are all non fraudulent transactions. A frequent use of an account is less likely to be caused by fraudsters. 

---

Let's see nameOrig with frequency = 2


In [ ]:
nameOrig_gt2 = nameOrig_counts[nameOrig_counts == 2].index
subset_2 = data[data['nameOrig'].isin(nameOrig_gt2)]
subset_2

Observations:

- The frequency displayed above shows that fraudulent transactions have max frequency = 1 but I'm checking and there are names that appear 2 times with fraudulent rows. There is another case of transactions that have 1 fraud record and 1 no fraud record. 

---

Let's see those rows. How many do we have?

In [ ]:
pickle_filename = 'fraud_and_nofraud_nameOrig.pkl'
pickle_filepath = os.path.join(pickle_directory, pickle_filename)

is_cached = False

try:
    with open(pickle_filepath, 'rb') as file:
        fraud_and_nofraud_nameOrig = pickle.load(file)
    is_cached = True
except FileNotFoundError:
    is_cached = False

if is_cached:
    print("The DataFrame is already stored in cache")
else:
    print("DataFrame not found. Calculating...")
    nameOrig_groups = subset_2.groupby('nameOrig')['isFraud'].unique()
    common_names = nameOrig_groups[nameOrig_groups.apply(lambda x: set(x) == {'fraud', 'no_fraud'})].index
    fraud_and_nofraud_nameOrig = subset_2[subset_2['nameOrig'].isin(common_names)]
    
    with open(pickle_filepath, 'wb') as file:
        pickle.dump(fraud_and_nofraud_nameOrig, file)

fraud_and_nofraud_nameOrig.head()


In [ ]:
print(f"Number of unique 'nameOrig' values in common_subset: {fraud_and_nofraud_nameOrig['nameOrig'].nunique()}")

Observations:

There are very few records with these characteristics to extract conclusions but this behaviour is different from the others so further analysis would be helpful as soon as more data would be created.

## oldbalanceOrig vs. isFraud

### Statistical tests

In [ ]:
group1 = data[data['isFraud'] == 'fraud']['oldbalanceOrig']
group2 = data[data['isFraud'] == 'no_fraud']['oldbalanceOrig']

stat, p_value = mannwhitneyu(group1, group2)

print(f"Mann-Whitney U Statistic: {stat}")
print(f"P-Value: {p_value}")

- p-value = 0 so there is a significative statistical difference of 'oldbalanceOrig' between the two catories of 'isFraud'. The variable 'oldbalanceOrig' behaves differently in each group.

- Mann-Whitney U Statistic: 42337964041.5. The range between both groups is very huge. 


### Summary stats

In [ ]:
data.groupby('isFraud')['oldbalanceOrig'].describe()

Observations:

- Non fraudulent category has a lower mean and a smaller range of values (from 0 to 43,818,856), showing that non-fraudulent transactions are more consistently lower in this feature.

- High variability in both categories

### Visualization

In [ ]:
contingency_table = pd.crosstab(data['oldbalanceOrig'], data['isFraud'])
contingency_table

In [ ]:
contingency_table[contingency_table['fraud'] > 2]

Observations: 

 - As seen in the 'amount' section, 1.000000e+07 is a commonly used value for fraud transactions. 

In [ ]:
filename = 'violinplot_oldbalanceOrig_vs_fraud.png'
plot_filepath = os.path.join(images_directory, filename)

if not os.path.isfile(plot_filepath):
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='isFraud', y='oldbalanceOrig', data=data, palette='pastel')
    plt.title('Violin Plot of oldbalanceOrig by isFraud')
    plt.tight_layout()

    plt.savefig(plot_filepath)
    plt.close() 

Image(filename=plot_filepath)

Observations: 

- The fraud violin plot shows much more outliers. The variability is very high.

- There is a small bump in the fraudulent transactions for the amount of 10M. Do fraudsters target those accounts?

### Additional statistics

In [ ]:
oldbalanceorig_0 = data[data['oldbalanceOrig'] == 0]
oldbalanceorig_0

In [ ]:
oldbalanceorig_0_fraud = oldbalanceorig_0[oldbalanceorig_0['isFraud'] == 'fraud']
oldbalanceorig_0_fraud.head()

In [ ]:
oldbalanceorig_0_no_fraud = oldbalanceorig_0[oldbalanceorig_0['isFraud'] == 'no_fraud']
oldbalanceorig_0_no_fraud.head()

In [ ]:
print(f'The amount of the old balances with 0€ is: {len(oldbalanceorig_0)}')
print(f'The percentage of the old balances with 0€ is: {(len(oldbalanceorig_0)/len(data)*100):.2f}%')

In [ ]:

print(f'The amount of the fraudulent transactions from accounts with 0€ is: {len(oldbalanceorig_0_fraud)}')
print(f'The amount of the NON fraudulent transactions from accounts with 0€ is: {len(oldbalanceorig_0_no_fraud)}')
print(f'The percentage of such NON fraudulent transactions with 0€ is: {(len(oldbalanceorig_0_no_fraud)/len(oldbalanceorig_0)*100):.4f}%')

Observations:

The 33% of accounts with transactions had balance = 0€ in the origin. From those, the vast majority are non fraudulent. I need more information here because I don't understand, taking into consideration that we don't have the type CREDIT, how users can operate with their accounts. Are they mistakes? 

## newbalanceOrig vs. isFraud

### Statistical tests

In [ ]:
group1 = data[data['isFraud'] == 'fraud']['newbalanceOrig']
group2 = data[data['isFraud'] == 'no_fraud']['newbalanceOrig']

stat, p_value = mannwhitneyu(group1, group2)

print(f"Mann-Whitney U Statistic: {stat}")
print(f"P-Value: {p_value}")

- p-value = 0 so there is a significative statistical difference of 'newbalanceOrig' between the two catories of 'isFraud'. The variable 'newbalanceOrig' behaves differently in each group.

- Mann-Whitney U Statistic: 15460387631.5. The range between both groups is very huge. 

### Summary stats

In [ ]:
data.groupby('isFraud')['newbalanceOrig'].describe()

Obsrvations:

Have you seen the quantiles? Fraudulent transactions are 0 until the 75th and then it reaches an even higher maximum value than the non fraudulent transactions. 

### Visualization

In [ ]:
contingency_table = pd.crosstab(data['newbalanceOrig'], data['isFraud'])
contingency_table

Observations: 

 - Again, as in oldbalanceOrig, 1.000000e+07 is a commonly used balance in fraud transactions. 

In [ ]:
contingency_table[contingency_table['fraud'] > 1]

In [ ]:
filename = 'violinplot_newbalanceOrig_vs_fraud.png'
plot_filepath = os.path.join(images_directory, filename)

if not os.path.isfile(plot_filepath):
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='isFraud', y='newbalanceOrig', data=data, palette='pastel')
    plt.title('Violin Plot of newbalanceOrig by isFraud')
    plt.tight_layout()

    plt.savefig(plot_filepath)
    plt.close()  

Image(filename=plot_filepath)

In [ ]:
filename = 'violinplot_newbalanceOrig_vs_fraud.png'
plot_filepath = os.path.join(images_directory, filename)

if not os.path.isfile(plot_filepath):
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='isFraud', y='newbalanceOrig', data=data, palette='pastel')
    plt.title('Violin Plot of newbalanceOrig by isFraud')
    plt.tight_layout()

    plt.savefig(plot_filepath)
    plt.close()  

Image(filename=plot_filepath)

Observations:

Outliers here are very extreme.

### Additional statistics

In [ ]:
newbalance_0 = data[data['newbalanceOrig'] == 0]

estudiar los balances con 0 euros 
balances con fraud puedo sacar algo de ahi?
what about create a column that is the difference between oldbalance and new balance?

## oldbalanceDest vs. isFraud

### Statistical tests

In [ ]:
group1 = data[data['isFraud'] == 'fraud']['oldbalanceDest']
group2 = data[data['isFraud'] == 'no_fraud']['oldbalanceDest']

stat, p_value = mannwhitneyu(group1, group2)

print(f"Mann-Whitney U Statistic: {stat}")
print(f"P-Value: {p_value}")

- p-value = 0 so there is a significative statistical difference of 'oldbalanceDest' between the two catories of 'isFraud'. The variable 'oldbalanceDest' behaves differently in each group.

- Mann-Whitney U Statistic: 19183643293.5. The range between both groups is very huge. 

### Summary stats

In [ ]:
data.groupby('isFraud')['oldbalanceDest'].describe()

Observations:

It's surprising the how much the fraudsters are actually using accounts with balance = 0€ in the destination (even the median is still 0!). Are those accounts that receive the money from fraudsters and that is the reason why they have no money? Could that be used as an alert? 

### Visualization

In [ ]:
contingency_table = pd.crosstab(data['oldbalanceDest'], data['isFraud'])
contingency_table

In [ ]:
contingency_table[contingency_table['fraud'] > 1]

Observations: 

 - Again, as in the previous balances, 1.000000e+07 is a commonly used balances in fraud transactions records

In [ ]:
filename = 'violinplot_oldbalanceDest_vs_fraud.png'
plot_filepath = os.path.join(images_directory, filename)

if not os.path.isfile(plot_filepath):
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='isFraud', y='oldbalanceDest', data=data, palette='pastel')
    plt.title('Violin Plot of oldbalanceDest by isFraud')
    plt.tight_layout()

    plt.savefig(plot_filepath)
    plt.close()  

Image(filename=plot_filepath)

Observations:

- Both violin plots are extremely flat but it's clear that the variability in balances for non fraudulent transactions is higher.

### Additional statistics

In [ ]:
oldbalancedest_0 = data[data['oldbalanceDest'] == 0]
oldbalancedest_0

In [ ]:
oldbalancedest_0_fraud = oldbalancedest_0[oldbalancedest_0['isFraud'] == 'fraud']
oldbalancedest_0_fraud.head()

In [ ]:
oldbalancedest_0_no_fraud = oldbalancedest_0[oldbalancedest_0['isFraud'] == 'no_fraud']
oldbalancedest_0_no_fraud

In [ ]:
print(f'The amount of the old balances with 0€ is: {len(oldbalancedest_0)}')
print(f'The percentage of the old balances with 0€ is: {(len(oldbalancedest_0)/len(data)*100):.2f}%')

In [ ]:

print(f'The amount of the fraudulent transactions from accounts with 0€ is: {len(oldbalancedest_0_fraud)}')
print(f'The amount of the NON fraudulent transactions from accounts with 0€ is: {len(oldbalancedest_0_no_fraud)}')
print(f'The percentage of such NON fraudulent transactions with 0€ is: {(len(oldbalancedest_0_no_fraud)/len(oldbalancedest_0)*100):.4f}%')

Observations:

The 42.5% of accounts with transactions had balance = 0€ in the destination. From those, the vast majority are non fraudulent. I need more information here because I don't understand, how can some transactions be payments but we don't see the amount being translated to the destination?


## newbalanceDest vs. isFraud

### Statistical tests

In [ ]:
group1 = data[data['isFraud'] == 'fraud']['newbalanceDest']
group2 = data[data['isFraud'] == 'no_fraud']['newbalanceDest']

stat, p_value = mannwhitneyu(group1, group2)

print(f"Mann-Whitney U Statistic: {stat}")
print(f"P-Value: {p_value}")

- p-value = 4.847058792178127e-39. It's not zero like the previous balances but still it's too low to be considered. So I can say that there is a significative statistical difference of 'newbalanceDest' between the two catories of 'isFraud'. The variable 'newbalanceDest' behaves differently in each group.

- Mann-Whitney U Statistic: 23982267534.5. The range between both groups is very huge. 

### Summary stats

In [ ]:
data.groupby('isFraud')['newbalanceDest'].describe()

Obervations:

- Means are very close.
- Fraud transactions tend to have a higher amount since the 50th quantile is much smaller than the no_fraud transactions. The range is also lower.

### Visualisation

In [ ]:
contingency_table = pd.crosstab(data['newbalanceDest'], data['isFraud'])
contingency_table

In [ ]:
contingency_table[contingency_table['fraud'] > 1]

Observations: 

 -  1.000000e+07 appears as a very important balance that is common in fraud çtransactions with balances

In [ ]:
filename = 'violinplot_newbalanceDest_vs_fraud.png'
plot_filepath = os.path.join(images_directory, filename)

if not os.path.isfile(plot_filepath):
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='isFraud', y='newbalanceDest', data=data, palette='pastel')
    plt.title('Violin Plot of newbalanceDest by isFraud')
    plt.tight_layout()

    plt.savefig(plot_filepath)
    plt.close()  

Image(filename=plot_filepath)

Observations:

The distribution of this variable is very similar to the distribution of the variable 'oldbalanceDest'

### Additional statistics

In [ ]:
newbalancedest_0 = data[data['newbalanceDest'] == 0]
newbalancedest_0

In [ ]:
newbalancedest_0_fraud = newbalancedest_0[newbalancedest_0['isFraud'] == 'fraud']
newbalancedest_0_fraud.head()

In [ ]:
newbalancedest_0_no_fraud = newbalancedest_0[newbalancedest_0['isFraud'] == 'no_fraud']
newbalancedest_0_no_fraud

In [ ]:
print(f'The amount of the old balances with 0€ is: {len(newbalancedest_0)}')
print(f'The percentage of the old balances with 0€ is: {(len(newbalancedest_0)/len(data)*100):.2f}%')

In [ ]:

print(f'The amount of the fraudulent transactions from accounts with 0€ is: {len(newbalancedest_0_fraud)}')
print(f'The amount of the NON fraudulent transactions from accounts with 0€ is: {len(newbalancedest_0_no_fraud)}')
print(f'The percentage of such NON fraudulent transactions with 0€ is: {(len(newbalancedest_0_no_fraud)/len(newbalancedest_0)*100):.4f}%')

Observations:

Both oldbalanceDest and newbalanceDest variables have very similar values and percentages related to values = 0€

## Creating variables for difference of balances 

In [ ]:
data['diffbalanceOrig'] = data['newbalanceOrig'] - data['oldbalanceOrig']
data['diffbalanceDest'] = data['newbalanceDest'] - data['oldbalanceDest']

In [ ]:
data

## Correlation

In [ ]:
data.dtypes

In [ ]:
numerical_cols = data.select_dtypes(include=['float32', 'int16', 'int8']).columns
categorical_cols = data.select_dtypes(include=['object', 'category']).columns



In [ ]:
pickle_filename = 'correlation_matrix.pkl'
pickle_filepath = os.path.join(pickle_directory, pickle_filename)

is_cached = False

try:
    correlation_matrix = load_from_pickle(pickle_filepath)
    is_cached = True
except FileNotFoundError:
    is_cached = False

if is_cached:
    print("The correlation matrix is stored in cache")
else:
    print("Correlation matrix not found. Calculating...")
    
    encoder = ce.TargetEncoder(cols=categorical_cols)
    target_col = numerical_cols[0]
    
    df_encoded = encoder.fit_transform(data, data[target_col])
    correlation_matrix = df_encoded.corr()

    save_to_pickle(correlation_matrix, pickle_filepath)

correlation_matrix


In [ ]:
heatmap_filename = 'heatmap_correlation_matrix.png'
heatmap_filepath = os.path.join(images_directory, heatmap_filename)

if not os.path.isfile(heatmap_filepath):
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
    plt.title('Heatmap of the Correlation Matrix')
    plt.tight_layout()
    plt.savefig(heatmap_filepath)
    plt.close()  

Image(filename=heatmap_filepath)

In [ ]:


threshold = 0.9

high_correlation_pairs = [(col1, col2) for col1 in correlation_matrix.columns for col2 in correlation_matrix.columns 
                          if col1 != col2 and abs(correlation_matrix.loc[col1, col2]) > threshold]

variables_to_remove = set()
for col1, col2 in high_correlation_pairs:
    if col1 not in variables_to_remove and col2 not in variables_to_remove:
        variables_to_remove.add(col2)
print(f"Variables to remove due to high correlation: {variables_to_remove}")

reduced_df = df_encoded.drop(columns=variables_to_remove)

reduced_correlation = reduced_df.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(reduced_correlation, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
plt.title('Heatmap of the Reduced Correlation Matrix')
plt.show()

In [ ]:
data

In [ ]:
data.to_parquet('../data/silver/df_fraud_bivar.parquet', engine= 'fastparquet')